In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\| **Optimization** \|\| [Save &
Load Model](saveloadrun_tutorial.html)

优化模型参数
===========================

有了模型和数据，就可以用训练、验证、测试等途径来优化模型的参数。训练模型是一个迭代过程；在每个迭代中模型猜测一次输出，在每个迭代中计算一次loss，收集对应参数猜测误差的导数，用梯度来优化参数。



In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

100%|██████████| 26.4M/26.4M [00:01<00:00, 17.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 272kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.50MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.3MB/s]


超参数
===============

超参数是控制模型优化过程的可调整参数。不同的超参数值可以影响模型的训练和收敛速度关于超参数控制  [更多](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)


训练过程的几个超参数：

- Epochs 在训练数据迭代的轮次
- Batch Size 一次传播中采样的数据数量
- Learning Rate 每个批次/轮次更新参数的程度。越小学习速度越慢，但过大的学习率会在训练过程中产生不可预测的行为


In [4]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

优化循环
=================

设置好超参数之后，可以通过优化循环来训练和优化模型。每次优化循环叫做一次**epoch**

每个 epoch 由两个主要部分构成：
- 训练循环：在训练集上迭代，尝试逼近最优参数
- 验证/测试循环：在测试集上迭代，检验模型表现是否更好


要简要熟悉这些训练循环的概念


损失函数
-----------
当输入训练数据时，未经训练的模型可能无法给出正确答案。损失函数用来衡量预测值和目标值大差异程度，此时就需要在训练过程中最小化损失函数。为了计算损失，用给定采样数据作为输入的预测值和正确的标签值进行比较


常见损失函数包括：
[nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)
(Mean Square Error) 用于回归任务。
[nn.NLLLoss](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss)
(Negative Log Likelihood) 用于分类任务。
[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss)
结合了`nn.LogSoftmax` 和 `nn.NLLLoss`。

把模型输出的 logits 输入 `nn.CrossEntropyLoss`，之后正则化 logits 并且计算预测误差

In [5]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

优化器
=========

优化是在训练步骤中调整模型参数来减少模型误差的步骤。优化算法定义了这个过程如何进行的（这个例子中使用了Stochastic Gradient Descent）。优化逻辑被封装在了`optimizer`对象中。这里用了SGD优化器。除此之外还有很多不同的[优化器](https://pytorch.org/docs/stable/optim.html)可以在Pytorch使用，比如ADAM和RMSProp在不同的数据和模型表现得更好

通过注册模型需要被训练的参数来初始化优化器，将其传入学习率的超参数

In [6]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

 在这个训练循环中，优化发生在三个步骤中：

- 调用 `optimizer.zero_grad()` 来重置模型参数的梯度。梯度是默认累加的，为了防止重复累加梯度，把每个迭代中把梯度置零了

- 通过调用`loss.backward()`来反向传播预测的损失值。PyTorch会存放每个参数对损失函数的梯度

- 有了反向传播获取的梯度之后，通过调用`optimizer.step()`来调整参数


完整实现
=========
优化模型代码上，我们定义`train_loop`。
定义`test_loop`来验证模型在测试数据上的表现

In [8]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    # 把模型设为训练模式以便批量归一化和dropout层的实现
    # 不必要但是是最佳实现
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # 计算预测及损失
        pred = model(X)
        loss = loss_fn(pred, y)

        # 反向传播
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):

    # 把模型设置为验证模式，同样利用批次归一化和dropout
    # 不必要，但是是最佳实践
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # 在 torch.no_grad() 下评测模型，来确保评测模式下没有梯度计算
    # 以减轻不必要的梯度计算来减少梯度计算和requires_grad=True的参数的内存使用
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

初始化损失函数和优化器，并将其传入`train_loop` 和 `test_loop`。增加 epochs 的次数来提高模型的表现


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.301517  [   64/60000]
loss: 2.288876  [ 6464/60000]
loss: 2.278125  [12864/60000]
loss: 2.272617  [19264/60000]
loss: 2.255259  [25664/60000]
loss: 2.237951  [32064/60000]
loss: 2.234985  [38464/60000]
loss: 2.210053  [44864/60000]
loss: 2.208609  [51264/60000]
loss: 2.181289  [57664/60000]
Test Error: 
 Accuracy: 53.5%, Avg loss: 2.177221 

Epoch 2
-------------------------------
loss: 2.189040  [   64/60000]
loss: 2.175742  [ 6464/60000]
loss: 2.131916  [12864/60000]
loss: 2.138632  [19264/60000]
loss: 2.096725  [25664/60000]
loss: 2.054349  [32064/60000]
loss: 2.060841  [38464/60000]
loss: 2.002714  [44864/60000]
loss: 2.012072  [51264/60000]
loss: 1.931599  [57664/60000]
Test Error: 
 Accuracy: 58.7%, Avg loss: 1.938022 

Epoch 3
-------------------------------
loss: 1.980018  [   64/60000]
loss: 1.941277  [ 6464/60000]
loss: 1.844003  [12864/60000]
loss: 1.859810  [19264/60000]
loss: 1.761654  [25664/60000]
loss: 1.726153  [32064/600

Further Reading
===============

-   [Loss
    Functions](https://pytorch.org/docs/stable/nn.html#loss-functions)
-   [torch.optim](https://pytorch.org/docs/stable/optim.html)
-   [Warmstart Training a
    Model](https://pytorch.org/tutorials/recipes/recipes/warmstarting_model_using_parameters_from_a_different_model.html)
